<a href="https://www.kaggle.com/code/smeyra/langchain-rag-metin-koleksiyonundan-bilgi-ekme?scriptVersionId=315026200" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# 1) KURULUM
# ============================================================
!pip install -q "langchain>=0.2" "langchain-community>=0.2" \
               "langchain-text-splitters>=0.2" \
               transformers accelerate sentencepiece \
               pypdf faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, 

In [2]:
"""import os
os._exit(00)"""

'import os\nos._exit(00)'

In [3]:
# ============================================================
# 2) İÇE AKTARIMLAR
# ============================================================
import os, textwrap, glob
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [4]:
# ============================================================
# 3) MODEL ve EMBEDDING
# ============================================================
# -- LLM (küçük ve hızlı): Qwen 0.5B Instruct
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # alternatif: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
    repetition_penalty=1.05,
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

# -- Embedding (hız/kalite dengesi iyi)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import os

In [6]:
# ============================================================
# 4) DOKÜMANLARI YÜKLEME
#    /kaggle/input/altinda-belge-klasoru/ içindeki .txt, .md, .pdf dosyaları oku
# ============================================================
DOC_DIR = "/kaggle/input/datasets/smeyra/ragdocs/ragdoc"   # <-- kendi klasör yolunuzu girin

# DirectoryLoader PDF ve düz metinleri otomatik tespit etmez; pattern ile iki kez çağırıyoruz:
docs = []
if os.path.isdir(DOC_DIR):
    # TXT & MD
    if glob.glob(os.path.join(DOC_DIR, "**/*.txt"), recursive=True) or \
       glob.glob(os.path.join(DOC_DIR, "**/*.md"), recursive=True):
        loader_txt = DirectoryLoader(DOC_DIR, glob="**/*.txt", loader_cls=TextLoader, recursive=True, show_progress=True)
        loader_md  = DirectoryLoader(DOC_DIR, glob="**/*.md",  loader_cls=TextLoader, recursive=True, show_progress=True)
        docs += loader_txt.load() + loader_md.load()
    # PDF
    for pdf_path in glob.glob(os.path.join(DOC_DIR, "**/*.pdf"), recursive=True):
        try:
            docs += PyPDFLoader(pdf_path).load()
        except Exception as e:
            print(f"PDF okunamadı: {pdf_path} | Hata: {e}")

else:
    raise FileNotFoundError(f"Klasör bulunamadı: {DOC_DIR}")

print(f"✅ Yüklenen doküman sayısı: {len(docs)}")


100%|██████████| 3/3 [00:00<00:00, 253.63it/s]
0it [00:00, ?it/s]

✅ Yüklenen doküman sayısı: 3


In [7]:
# ============================================================
# 5) PARÇALAMA (Chunking)
# ============================================================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # içerik yapısına göre 500-1200 arası önerilir
    chunk_overlap=120,   # bağlam taşımak için küçük örtüşme
    separators=["\n\n", "\n", " ", ""]
)
chunks = splitter.split_documents(docs)
print(f"✅ Üretilen parça sayısı: {len(chunks)}")


✅ Üretilen parça sayısı: 4


In [8]:
# ============================================================
# 6) VEKTÖR VERİTABANI (FAISS) — OLUŞTUR & KAYDET
# ============================================================
INDEX_DIR = "/kaggle/working/faiss_index"

vectordb = FAISS.from_documents(chunks, embeddings)
os.makedirs(INDEX_DIR, exist_ok=True)
vectordb.save_local(INDEX_DIR)
print(f"✅ FAISS index kaydedildi: {INDEX_DIR}")

# (İsteyenler için: Sonradan yükleme)
# vectordb = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

retriever = vectordb.as_retriever(search_kwargs={"k": 4})

✅ FAISS index kaydedildi: /kaggle/working/faiss_index


In [9]:
# ============================================================
# 7) PROMPT ve RAG ZİNCİRİ
# ============================================================
prompt = PromptTemplate.from_template(textwrap.dedent("""
    Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) veriliyor.
    Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile yanıt ver.
    Bağlamda olmayan bilgi için "Metinde bu bilgi yer almıyor." de; uydurma yapma.

    Soru: {question}

    Bağlam:
    {context}

    Cevap:
"""))

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # kısa/orta context için yeterli
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

In [10]:
# ============================================================
# 8) SORU ÇALIŞTIR
# ============================================================
question = "Belgelerdeki ana problem tanımı nedir ve önerilen çözüm yaklaşımı nasıl özetlenmiştir?"
result = qa_chain({"query": question})

print("\n=== SORU ===\n", question)
print("\n=== YANIT ===\n", result["result"])

print("\n=== KAYNAK PARÇALAR ===")
for i, d in enumerate(result["source_documents"], 1):
    meta = d.metadata
    src = meta.get("source", "NA")
    page = meta.get("page", "NA")
    print(f"\n--- Parça {i} | Kaynak: {src} | Sayfa: {page} ---")
    print(textwrap.shorten(d.page_content.replace("\n", " "), width=300, placeholder="..."))

/tmp/ipykernel_16/2011953693.py:5: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa_chain({"query": question})
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== SORU ===
 Belgelerdeki ana problem tanımı nedir ve önerilen çözüm yaklaşımı nasıl özetlenmiştir?

=== YANIT ===
 
Aşağıda bir kullanıcı sorusu ve belgeden getirilen ilgili parçalar (context) veriliyor.
Sadece bu bağlamdan yararlanarak, akademik ve açık bir Türkçe ile yanıt ver.
Bağlamda olmayan bilgi için "Metinde bu bilgi yer almıyor." de; uydurma yapma.

Soru: Belgelerdeki ana problem tanımı nedir ve önerilen çözüm yaklaşımı nasıl özetlenmiştir?

Bağlam:
Operating the Climate Control System  Your Googlecar has a climate control system that allows you to adjust the temperature and airflow in the car. To operate the climate control system, use the buttons and knobs located on the center console.  Temperature: The temperature knob controls the temperature inside the car. Turn the knob clockwise to increase the temperature or counterclockwise to decrease the temperature. Airflow: The airflow knob controls the amount of airflow inside the car. Turn the knob clockwise to increase the 